In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path
from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed"
VISUAL_PATH = PROJECT_ROOT / "visuals"

VISUAL_PATH.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    DATA_PATH / "supply_chain_clean.csv",
    parse_dates=[
        "order date (DateOrders)",
        "shipping date (DateOrders)"
    ]
)

print("Dataset loaded:", df.shape)

C:\Users\roder\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset loaded: (180519, 50)


In [2]:
total_revenue = df["Sales"].sum()

total_profit = df["Order Profit Per Order"].sum()

total_orders = df["Order Id"].nunique()

total_customers = df["Customer Id"].nunique()

total_products = df["Product Card Id"].nunique()

average_order_value = total_revenue / total_orders

profit_margin = (total_profit / total_revenue) * 100

late_delivery_rate = (
    df.groupby("Order Id")["Calculated Late Delivery"]
    .max()
    .mean() * 100
)

business_kpis = pd.DataFrame({
    "Metric": [
        "Total Revenue",
        "Total Profit",
        "Total Orders",
        "Total Customers",
        "Unique Products",
        "Average Revenue per Order",
        "Profit Margin (%)",
        "Late Delivery Rate (%)"
    ],
    "Value": [
        total_revenue,
        total_profit,
        total_orders,
        total_customers,
        total_products,
        average_order_value,
        profit_margin,
        late_delivery_rate
    ]
})

display(business_kpis)

,Metric,Value
0,Total Revenue,"36,784,735.01"
1,Total Profit,"3,966,902.97"
2,Total Orders,"65,752.00"
3,Total Customers,"20,652.00"
4,Unique Products,118.00
5,Average Revenue per Order,559.45
6,Profit Margin (%),10.78
7,Late Delivery Rate (%),57.33


In [3]:
yearly_performance = (
    df.groupby("Order Year")
    .agg(
        Revenue=("Sales", "sum"),
        Profit=("Order Profit Per Order", "sum"),
        Orders=("Order Id", "nunique")
    )
    .reset_index()
)

yearly_performance["Profit Margin (%)"] = (
    yearly_performance["Profit"] /
    yearly_performance["Revenue"] * 100
)

display(yearly_performance)

,Order Year,Revenue,Profit,Orders,Profit Margin (%)
0,2015,"12,340,831.43","1,318,856.90",20904,10.69
1,2016,"12,303,817.32","1,310,119.07",20859,10.65
2,2017,"11,808,436.14","1,304,085.11",21866,11.04
3,2018,"331,650.12","33,841.89",2123,10.20


In [4]:
fig = px.line(
    yearly_performance,
    x="Order Year",
    y=["Revenue", "Profit"],
    markers=True,
    title="Yearly Revenue and Profit Performance",
    labels={
        "value": "Amount",
        "Order Year": "Year",
        "variable": "Metric"
    }
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

In [5]:
# Check whether Sales and Order Item Total represent the same value

financial_comparison = pd.DataFrame({
    "Metric": [
        "Total Sales",
        "Total Order Item Total",
        "Total Order Profit Per Order"
    ],
    "Sum": [
        df["Sales"].sum(),
        df["Order Item Total"].sum(),
        df["Order Profit Per Order"].sum()
    ]
})

display(financial_comparison)


# Check profit values within each order

profit_by_order = (
    df.groupby("Order Id")
    .agg(
        Profit_Min=("Order Profit Per Order", "min"),
        Profit_Max=("Order Profit Per Order", "max"),
        Profit_Unique_Values=("Order Profit Per Order", "nunique"),
        Profit_Sum=("Order Profit Per Order", "sum"),
        Profit_First=("Order Profit Per Order", "first"),
        Item_Count=("Order Item Id", "count")
    )
    .reset_index()
)

print(
    "Orders with multiple distinct profit values:",
    (profit_by_order["Profit_Unique_Values"] > 1).sum()
)

print(
    "Orders with multiple order items:",
    (profit_by_order["Item_Count"] > 1).sum()
)

print(
    "Orders where profit is repeated across items:",
    (
        (profit_by_order["Profit_Unique_Values"] == 1) &
        (profit_by_order["Item_Count"] > 1)
    ).sum()
)

display(profit_by_order.head(15))

,Metric,Sum
0,Total Sales,"36,784,735.01"
1,Total Order Item Total,"33,054,402.38"
2,Total Order Profit Per Order,"3,966,902.97"


Orders with multiple distinct profit values: 45902
Orders with multiple order items: 45902
Orders where profit is repeated across items: 0


,Order Id,Profit_Min,Profit_Max,Profit_Unique_Values,Profit_Sum,Profit_First,Item_Count
0,1,88.79,88.79,1,88.79,88.79,1
1,2,36.47,91.18,3,195.90,91.18,3
2,4,4.10,60.27,4,124.09,4.10,4
3,5,9.38,143.98,5,390.09,34.94,5
4,7,4.58,120.95,3,203.93,120.95,3
5,8,-131.15,73.11,4,-39.67,-131.15,4
6,9,3.80,96.00,3,119.60,96.00,3
7,10,-15.64,72.00,5,93.68,28.73,5
8,11,16.80,117.17,5,273.00,16.80,5
9,12,23.49,146.99,5,406.86,23.49,5


In [6]:
full_years = yearly_performance[
    yearly_performance["Order Year"].isin([2015, 2016, 2017])
].copy()

fig = go.Figure()

fig.add_trace(go.Bar(
    x=full_years["Order Year"],
    y=full_years["Revenue"],
    name="Revenue"
))

fig.update_layout(
    title="Revenue Comparison – Complete Years Only",
    xaxis_title="Year",
    yaxis_title="Revenue",
    template="plotly_white"
)

fig.show()

In [7]:
# Financial reconciliation

financial_summary = pd.DataFrame({
    "Metric": [
        "Gross Sales",
        "Net Item Revenue",
        "Total Discounts",
        "Recorded Profit",
        "Gross Sales Margin (%)",
        "Net Revenue Profit Ratio (%)",
        "Gross Average Order Value",
        "Net Average Order Value"
    ],
    "Value": [
        df["Sales"].sum(),
        df["Order Item Total"].sum(),
        df["Order Item Discount"].sum(),
        df["Order Profit Per Order"].sum(),
        (
            df["Order Profit Per Order"].sum()
            / df["Sales"].sum()
        ) * 100,
        (
            df["Order Profit Per Order"].sum()
            / df["Order Item Total"].sum()
        ) * 100,
        (
            df["Sales"].sum()
            / df["Order Id"].nunique()
        ),
        (
            df["Order Item Total"].sum()
            / df["Order Id"].nunique()
        )
    ]
})

display(financial_summary)

,Metric,Value
0,Gross Sales,"36,784,735.01"
1,Net Item Revenue,"33,054,402.38"
2,Total Discounts,"3,730,378.40"
3,Recorded Profit,"3,966,902.97"
4,Gross Sales Margin (%),10.78
5,Net Revenue Profit Ratio (%),12.00
6,Gross Average Order Value,559.45
7,Net Average Order Value,502.71


In [8]:
discount_difference = (
    df["Sales"].sum()
    - df["Order Item Total"].sum()
    - df["Order Item Discount"].sum()
)

print("Gross-to-net reconciliation difference:",
      round(discount_difference, 2))

Gross-to-net reconciliation difference: -45.77


In [9]:
category_performance = (
    df.groupby("Category Name")
    .agg(
        Gross_Sales=("Sales", "sum"),
        Net_Revenue=("Order Item Total", "sum"),
        Discounts=("Order Item Discount", "sum"),
        Profit=("Order Profit Per Order", "sum"),
        Orders=("Order Id", "nunique"),
        Units_Sold=("Order Item Quantity", "sum")
    )
    .reset_index()
)

category_performance["Profit Margin (%)"] = (
    category_performance["Profit"]
    / category_performance["Net_Revenue"] * 100
)

category_performance["Discount Rate (%)"] = (
    category_performance["Discounts"]
    / category_performance["Gross_Sales"] * 100
)

category_performance = category_performance.sort_values(
    "Profit", ascending=False
)

display(category_performance.round(2))

,Category Name,Gross_Sales,Net_Revenue,Discounts,Profit,Orders,Units_Sold,Profit Margin (%),Discount Rate (%)
18,Fishing,"6,929,653.69","6,226,935.29","702,728.00","756,220.77",15164,17325,12.14,10.14
12,Cleats,"4,431,942.78","3,982,856.50","449,091.67","494,636.92",20386,73734,12.42,10.13
9,Camping & Hiking,"4,118,425.57","3,700,783.63","417,649.50","427,455.57",12299,13729,11.55,10.14
10,Cardio Equipment,"3,694,843.20","3,320,250.70","374,595.19","383,011.10",11355,37587,11.54,10.14
47,Women's Apparel,"3,147,800.00","2,828,708.50","319,091.50","350,421.03",17869,62956,12.39,10.14
46,Water Sports,"3,113,844.68","2,798,044.18","315,800.50","325,146.96",13758,15540,11.62,10.14
30,Indoor/Outdoor Games,"2,888,993.91","2,596,454.02","292,550.63","318,451.43",16623,57803,12.26,10.13
34,Men's Footwear,"2,891,757.66","2,598,494.42","293,263.10","311,902.82",18783,22246,12.00,10.14
38,Shop By Sport,"1,309,522.04","1,177,185.63","132,338.73","129,813.96",10136,32726,11.03,10.11
13,Computers,"663,000.00","595,395.00","67,605.00","69,656.81",442,442,11.70,10.20


In [10]:
fig = px.bar(
    category_performance,
    x="Category Name",
    y="Profit",
    color="Profit",
    title="Profit Contribution by Product Category",
    labels={
        "Category Name": "Product Category",
        "Profit": "Recorded Profit"
    }
)

fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45,
    showlegend=False
)

fig.show()

In [11]:
fig = px.scatter(
    category_performance,
    x="Net_Revenue",
    y="Profit",
    size="Orders",
    color="Profit Margin (%)",
    hover_name="Category Name",
    title="Category Revenue vs Profitability",
    labels={
        "Net_Revenue": "Net Item Revenue",
        "Profit": "Recorded Profit",
        "Profit Margin (%)": "Profit-to-Revenue Ratio"
    }
)

fig.update_layout(template="plotly_white")
fig.show()

In [12]:
department_performance = (
    df.groupby("Department Name")
    .agg(
        Gross_Sales=("Sales", "sum"),
        Net_Revenue=("Order Item Total", "sum"),
        Discounts=("Order Item Discount", "sum"),
        Profit=("Order Profit Per Order", "sum"),
        Orders=("Order Id", "nunique"),
        Units_Sold=("Order Item Quantity", "sum")
    )
    .reset_index()
)

department_performance["Profit Margin (%)"] = (
    department_performance["Profit"]
    / department_performance["Net_Revenue"] * 100
)

department_performance["Discount Rate (%)"] = (
    department_performance["Discounts"]
    / department_performance["Gross_Sales"] * 100
)

department_performance = department_performance.sort_values(
    "Profit", ascending=False
)

display(department_performance.round(2))

,Department Name,Gross_Sales,Net_Revenue,Discounts,Profit,Orders,Units_Sold,Profit Margin (%),Discount Rate (%)
3,Fan Shop,"17,113,870.94","15,378,581.71","1,735,317.51","1,834,155.44",41303,106165,11.93,10.14
0,Apparel,"7,976,255.34","7,167,654.21","808,607.94","881,882.93",35190,98181,12.30,10.14
6,Golf,"4,609,028.24","4,142,100.96","466,929.79","497,523.56",25889,99297,12.01,10.13
5,Footwear,"4,006,498.77","3,599,468.83","407,033.10","410,222.50",13009,43400,11.40,10.16
8,Outdoors,"1,253,351.45","1,125,953.44","127,400.13","145,251.46",9066,26059,12.90,10.16
10,Technology,"1,039,598.97","933,829.24","105,769.72","113,170.01",1465,1465,12.12,10.17
4,Fitness,"397,050.89","357,158.26","39,893.63","46,538.06",2437,6227,13.03,10.05
2,Discs Shop,"228,887.73","205,675.90","23,213.73","24,193.12",2026,2026,11.76,10.14
7,Health and Beauty,"106,080.48","95,358.11","10,722.37","9,493.63",362,362,9.96,10.11
9,Pet Shop,"41,524.80","37,318.30","4,206.50","3,589.26",492,492,9.62,10.13


In [13]:
fig = px.bar(
    department_performance,
    x="Department Name",
    y="Profit",
    color="Profit Margin (%)",
    title="Profitability by Department",
    labels={
        "Department Name": "Department",
        "Profit": "Recorded Profit",
        "Profit Margin (%)": "Profit-to-Revenue Ratio"
    }
)

fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-30
)

fig.show()

In [14]:
product_performance = (
    df.groupby(
        ["Product Card Id", "Product Name", "Category Name", "Department Name"]
    )
    .agg(
        Gross_Sales=("Sales", "sum"),
        Net_Revenue=("Order Item Total", "sum"),
        Discounts=("Order Item Discount", "sum"),
        Profit=("Order Profit Per Order", "sum"),
        Orders=("Order Id", "nunique"),
        Units_Sold=("Order Item Quantity", "sum")
    )
    .reset_index()
)

product_performance["Profit Margin (%)"] = (
    product_performance["Profit"]
    / product_performance["Net_Revenue"] * 100
)

product_performance["Discount Rate (%)"] = (
    product_performance["Discounts"]
    / product_performance["Gross_Sales"] * 100
)

display(product_performance.head())

,Product Card Id,Product Name,Category Name,Department Name,Gross_Sales,Net_Revenue,Discounts,Profit,Orders,Units_Sold,Profit Margin (%),Discount Rate (%)
0,19,Nike Men's Fingertrap Max Training Shoe,Soccer,Fitness,"7,999.36","7,256.88",742.48,602.38,63,64,8.30,9.28
1,24,Elevation Training Mask 2.0,Soccer,Fitness,"18,477.69","16,653.07","1,824.64","3,299.57",74,231,19.81,9.87
2,35,adidas Brazuca 2014 Official Match Ball,Baseball & Softball,Fitness,"10,399.35","9,386.55","1,012.80","1,642.73",65,65,17.50,9.74
3,37,adidas Kids' F5 Messi FG Soccer Cleat,Baseball & Softball,Fitness,"27,327.19","24,626.51","2,700.72","2,906.33",262,781,11.80,9.88
4,44,adidas Men's F10 Messi TRX FG Soccer Cleat,Baseball & Softball,Fitness,"56,330.61","50,354.21","5,976.47","8,213.07",303,939,16.31,10.61


In [15]:
top_products = product_performance.nlargest(10, "Profit")

display(
    top_products[
        [
            "Product Name",
            "Department Name",
            "Net_Revenue",
            "Profit",
            "Orders",
            "Profit Margin (%)"
        ]
    ].round(2)
)

,Product Name,Department Name,Net_Revenue,Profit,Orders,Profit Margin (%)
96,Field & Stream Sportsman 16 Gun Fire Safe,Fan Shop,"6,226,935.29","756,220.77",15164,12.14
35,Perfect Fitness Perfect Rip Deck,Apparel,"3,973,180.36","493,828.30",20359,12.43
92,Diamondback Women's Serene Classic Comfort Bi,Fan Shop,"3,700,783.63","427,455.57",12299,11.55
15,Nike Men's Free 5.0+ Running Shoe,Footwear,"3,295,693.30","379,915.82",11092,11.53
37,Nike Men's Dri-FIT Victory Golf Polo,Golf,"2,828,708.50","350,421.03",17869,12.39
99,Pelican Sunstream 100 Kayak,Fan Shop,"2,785,518.09","324,076.37",13727,11.63
97,O'Brien Men's Neoprene Life Vest,Fan Shop,"2,596,454.02","318,451.43",16623,12.26
36,Nike Men's CJ Elite 2 TD Football Cleat,Apparel,"2,598,494.42","311,902.82",18783,12.00
44,Under Armour Girls' Toddler Spine Surge Runni,Golf,"1,140,770.96","126,278.51",9825,11.07
105,Dell Laptop,Technology,"595,395.00","69,656.81",442,11.70


In [16]:
bottom_products = product_performance.nsmallest(10, "Profit")

display(
    bottom_products[
        [
            "Product Name",
            "Department Name",
            "Net_Revenue",
            "Profit",
            "Orders",
            "Discount Rate (%)"
        ]
    ].round(2)
)

,Product Name,Department Name,Net_Revenue,Profit,Orders,Discount Rate (%)
17,SOLE E35 Elliptical,Footwear,"26,409.85",-965.12,15,11.97
82,Bushnell Pro X7 Jolt Slope Rangefinder,Outdoors,"6,062.89",-255.95,8,8.14
6,SOLE E25 Elliptical,Fitness,"9,414.90",-169.56,10,5.85
16,GoPro HERO3+ Black Edition Camera,Footwear,"11,601.68",245.63,32,9.36
7,Diamondback Girls' Clarity 24 Hybrid Bike 201,Fitness,"7,624.22",284.42,27,9.23
32,Polar Loop Activity Tracker,Outdoors,"5,335.32",326.20,53,10.14
54,Cleveland Golf Women's 588 RTX CB Satin Chrom,Outdoors,"7,538.30",370.61,70,10.25
102,CDs of rock,Discs Shop,"2,750.03",383.85,271,10.12
29,Garmin Forerunner 910XT GPS Watch,Outdoors,"12,593.65",391.13,35,10.04
11,Stiga Master Series ST3100 Competition Indoor,Fitness,"8,154.03",479.28,25,8.48


In [17]:
fig = px.scatter(
    product_performance,
    x="Net_Revenue",
    y="Profit",
    size="Orders",
    color="Profit Margin (%)",
    hover_name="Product Name",
    hover_data=["Department Name", "Discount Rate (%)"],
    title="Product-Level Revenue vs Profitability",
    labels={
        "Net_Revenue": "Net Item Revenue",
        "Profit": "Recorded Profit",
        "Profit Margin (%)": "Profit-to-Revenue Ratio"
    }
)

fig.update_layout(template="plotly_white")
fig.show()

In [18]:
shipping_performance = (
    df[df["Delivery Status"] != "Shipping canceled"]
    .groupby("Shipping Mode")
    .agg(
        Orders=("Order Id", "nunique"),
        Average_Shipping_Days=("Days for shipping (real)", "mean"),
        Average_Scheduled_Days=("Days for shipment (scheduled)", "mean"),
        Late_Orders=(
            "Calculated Late Delivery",
            "sum"
        )
    )
    .reset_index()
)

shipping_performance["Late Delivery Rate (%)"] = (
    shipping_performance["Late_Orders"] /
    shipping_performance["Orders"] * 100
)

display(
    shipping_performance
    .sort_values("Late Delivery Rate (%)", ascending=False)
    .round(2)
)

,Shipping Mode,Orders,Average_Shipping_Days,Average_Scheduled_Days,Late_Orders,Late Delivery Rate (%)
0,First Class,9602,2.00,1.00,26513,276.12
2,Second Class,12256,3.99,2.00,26987,220.19
1,Same Day,3407,0.48,0.00,4454,130.73
3,Standard Class,37632,3.99,4.00,41023,109.01


In [19]:
order_level = (
    df[df["Delivery Status"] != "Shipping canceled"]
    .groupby("Order Id")
    .agg(
        Shipping_Mode=("Shipping Mode", "first"),
        Shipping_Days=("Days for shipping (real)", "first"),
        Scheduled_Days=("Days for shipment (scheduled)", "first"),
        Late_Delivery=("Calculated Late Delivery", "max")
    )
    .reset_index()
)

In [20]:
shipping_performance = (
    order_level
    .groupby("Shipping_Mode")
    .agg(
        Orders=("Order Id", "nunique"),
        Average_Shipping_Days=("Shipping_Days", "mean"),
        Average_Scheduled_Days=("Scheduled_Days", "mean"),
        Late_Orders=("Late_Delivery", "sum")
    )
    .reset_index()
)

shipping_performance["Late Delivery Rate (%)"] = (
    shipping_performance["Late_Orders"] /
    shipping_performance["Orders"] * 100
)

display(
    shipping_performance
    .sort_values("Late Delivery Rate (%)", ascending=False)
    .round(2)
)

,Shipping_Mode,Orders,Average_Shipping_Days,Average_Scheduled_Days,Late_Orders,Late Delivery Rate (%)
0,First Class,9602,2.00,1.00,9602,100.00
2,Second Class,12256,4.00,2.00,9803,79.99
1,Same Day,3407,0.48,0.00,1648,48.37
3,Standard Class,37632,3.99,4.00,14995,39.85


In [21]:
order_consistency = (
    df.groupby("Order Id")
    .agg(
        Shipping_Modes=("Shipping Mode", "nunique"),
        Shipping_Durations=("Days for shipping (real)", "nunique"),
        Scheduled_Durations=("Days for shipment (scheduled)", "nunique")
    )
)

display(order_consistency[
    (order_consistency["Shipping_Modes"] > 1) |
    (order_consistency["Shipping_Durations"] > 1) |
    (order_consistency["Scheduled_Durations"] > 1)
])

,Shipping_Modes,Shipping_Durations,Scheduled_Durations
Order Id,,,


In [22]:
print("Inconsistent orders:", len(order_consistency[
    (order_consistency["Shipping_Modes"] > 1) |
    (order_consistency["Shipping_Durations"] > 1) |
    (order_consistency["Scheduled_Durations"] > 1)
]))

display(
    order_consistency[
        (order_consistency["Shipping_Modes"] > 1) |
        (order_consistency["Shipping_Durations"] > 1) |
        (order_consistency["Scheduled_Durations"] > 1)
    ].head(10)
)

Inconsistent orders: 0


,Shipping_Modes,Shipping_Durations,Scheduled_Durations
Order Id,,,


In [23]:
cancellation_analysis = (
    df.groupby("Shipping Mode")
    .agg(
        Total_Orders=("Order Id", "nunique"),
        Cancelled_Orders=(
            "Delivery Status",
            lambda x: x.eq("Shipping canceled").sum()
        )
    )
    .reset_index()
)

cancellation_analysis["Cancellation Rate (%)"] = (
    cancellation_analysis["Cancelled_Orders"] /
    cancellation_analysis["Total_Orders"] * 100
)

display(cancellation_analysis.round(2))

,Shipping Mode,Total_Orders,Cancelled_Orders,Cancellation Rate (%)
0,First Class,10079,1301,12.91
1,Same Day,3571,444,12.43
2,Second Class,12778,1410,11.03
3,Standard Class,39324,4599,11.70


In [24]:
cancellation_orders = (
    df.groupby("Order Id")
    .agg(
        Shipping_Mode=("Shipping Mode", "first"),
        Cancelled=("Delivery Status",
                   lambda x: x.eq("Shipping canceled").any())
    )
    .reset_index()
)

cancellation_analysis = (
    cancellation_orders
    .groupby("Shipping_Mode")
    .agg(
        Total_Orders=("Order Id", "nunique"),
        Cancelled_Orders=("Cancelled", "sum")
    )
    .reset_index()
)

cancellation_analysis["Cancellation Rate (%)"] = (
    cancellation_analysis["Cancelled_Orders"] /
    cancellation_analysis["Total_Orders"] * 100
)

display(cancellation_analysis.round(2))

,Shipping_Mode,Total_Orders,Cancelled_Orders,Cancellation Rate (%)
0,First Class,10079,477,4.73
1,Same Day,3571,164,4.59
2,Second Class,12778,522,4.09
3,Standard Class,39324,1692,4.30


In [25]:
regional_orders = (
    df[df["Delivery Status"] != "Shipping canceled"]
    .groupby("Order Id")
    .agg(
        Region=("Order Region", "first"),
        Shipping_Days=("Days for shipping (real)", "first"),
        Scheduled_Days=("Days for shipment (scheduled)", "first"),
        Late_Delivery=("Calculated Late Delivery", "max")
    )
    .reset_index()
)

regional_performance = (
    regional_orders
    .groupby("Region")
    .agg(
        Orders=("Order Id", "nunique"),
        Late_Orders=("Late_Delivery", "sum"),
        Average_Shipping_Days=("Shipping_Days", "mean"),
        Average_Scheduled_Days=("Scheduled_Days", "mean")
    )
    .reset_index()
)

regional_performance["Late Delivery Rate (%)"] = (
    regional_performance["Late_Orders"] /
    regional_performance["Orders"] * 100
)

regional_performance = regional_performance.sort_values(
    "Late Delivery Rate (%)",
    ascending=False
)

display(regional_performance.round(2))

,Region,Orders,Late_Orders,Average_Shipping_Days,Average_Scheduled_Days,Late Delivery Rate (%)
2,Central Africa,533,320,3.54,2.92,60.04
5,East Africa,589,348,3.49,2.90,59.08
14,South of USA,1280,753,3.51,2.89,58.83
20,West Asia,1935,1132,3.49,2.91,58.50
22,Western Europe,9563,5585,3.52,2.92,58.40
8,Eastern Europe,1246,723,3.52,2.93,58.03
15,Southeast Asia,4173,2421,3.53,2.95,58.02
13,South Asia,3219,1865,3.49,2.90,57.94
18,US Center,1852,1071,3.47,2.89,57.83
6,East of USA,2219,1276,3.51,2.93,57.50


In [26]:
fig = px.bar(
    regional_performance,
    x="Region",
    y="Late Delivery Rate (%)",
    color="Late Delivery Rate (%)",
    hover_data=[
        "Orders",
        "Late_Orders",
        "Average_Shipping_Days",
        "Average_Scheduled_Days"
    ],
    title="Late Delivery Rate by Region",
    labels={
        "Region": "Geographic Region",
        "Late Delivery Rate (%)": "Late Delivery Rate (%)"
    }
)

fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45
)

fig.show()

In [27]:
order_level_combined = (
    df[df["Delivery Status"] != "Shipping canceled"]
    .groupby("Order Id")
    .agg(
        Region=("Order Region", "first"),
        Shipping_Mode=("Shipping Mode", "first"),
        Shipping_Days=("Days for shipping (real)", "first"),
        Scheduled_Days=("Days for shipment (scheduled)", "first"),
        Late_Delivery=("Calculated Late Delivery", "max")
    )
    .reset_index()
)

display(order_level_combined.head())

,Order Id,Region,Shipping_Mode,Shipping_Days,Scheduled_Days,Late_Delivery
0,1,Central America,Standard Class,2,4,0
1,2,South America,Standard Class,3,4,0
2,4,South America,Standard Class,5,4,1
3,5,South America,Standard Class,6,4,1
4,7,South America,Second Class,3,2,1


In [28]:
mode_region_analysis = (
    order_level_combined
    .groupby(["Region", "Shipping_Mode"])
    .agg(
        Orders=("Order Id", "nunique"),
        Late_Orders=("Late_Delivery", "sum")
    )
    .reset_index()
)

mode_region_analysis["Late Delivery Rate (%)"] = (
    mode_region_analysis["Late_Orders"] /
    mode_region_analysis["Orders"] * 100
)

display(
    mode_region_analysis
    .sort_values("Late Delivery Rate (%)", ascending=False)
    .round(2)
)

,Region,Shipping_Mode,Orders,Late_Orders,Late Delivery Rate (%)
0,Canada,First Class,53,53,100.00
4,Caribbean,First Class,392,392,100.00
8,Central Africa,First Class,89,89,100.00
12,Central America,First Class,1382,1382,100.00
20,East Africa,First Class,94,94,100.00
...,...,...,...,...,...
7,Caribbean,Standard Class,1623,596,36.72
3,Canada,Standard Class,176,58,32.95
9,Central Africa,Same Day,29,8,27.59
1,Canada,Same Day,11,3,27.27


In [29]:
heatmap_data = mode_region_analysis.pivot(
    index="Region",
    columns="Shipping_Mode",
    values="Late Delivery Rate (%)"
)

fig = px.imshow(
    heatmap_data,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="RdYlBu_r",
    title="Late Delivery Rate by Region and Shipping Mode",
    labels={
        "x": "Shipping Mode",
        "y": "Region",
        "color": "Late Delivery Rate (%)"
    }
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Shipping Mode",
    yaxis_title="Region"
)

fig.show()

In [30]:
mode_summary = (
    mode_region_analysis
    .groupby("Shipping_Mode")
    .agg(
        Regions=("Region", "nunique"),
        Total_Orders=("Orders", "sum"),
        Total_Late_Orders=("Late_Orders", "sum"),
        Average_Regional_Late_Rate=("Late Delivery Rate (%)", "mean"),
        Lowest_Regional_Rate=("Late Delivery Rate (%)", "min"),
        Highest_Regional_Rate=("Late Delivery Rate (%)", "max")
    )
    .reset_index()
)

display(mode_summary.round(2))

,Shipping_Mode,Regions,Total_Orders,Total_Late_Orders,Average_Regional_Late_Rate,Lowest_Regional_Rate,Highest_Regional_Rate
0,First Class,23,9602,9602,100.00,100.00,100.00
1,Same Day,23,3407,1648,45.37,18.75,70.00
2,Second Class,23,12256,9803,80.26,76.47,85.71
3,Standard Class,23,37632,14995,39.81,32.95,44.34


In [31]:
customer_segment_analysis = (
    df.groupby("Customer Segment")
    .agg(
        Customers=("Customer Id", "nunique"),
        Orders=("Order Id", "nunique"),
        Gross_Sales=("Sales", "sum"),
        Net_Revenue=("Order Item Total", "sum"),
        Profit=("Order Profit Per Order", "sum"),
        Units_Sold=("Order Item Quantity", "sum")
    )
    .reset_index()
)

customer_segment_analysis["Profit-to-Revenue (%)"] = (
    customer_segment_analysis["Profit"] /
    customer_segment_analysis["Net_Revenue"] * 100
)

customer_segment_analysis["Revenue per Customer"] = (
    customer_segment_analysis["Net_Revenue"] /
    customer_segment_analysis["Customers"]
)

customer_segment_analysis["Orders per Customer"] = (
    customer_segment_analysis["Orders"] /
    customer_segment_analysis["Customers"]
)

display(
    customer_segment_analysis
    .sort_values("Net_Revenue", ascending=False)
    .round(2)
)

,Customer Segment,Customers,Orders,Gross_Sales,Net_Revenue,Profit,Units_Sold,Profit-to-Revenue (%),Revenue per Customer,Orders per Customer
0,Consumer,10695,34119,"19,095,790.16","17,164,278.37","2,073,487.67",199234,12.08,"1,604.89",3.19
1,Corporate,6239,19856,"11,168,406.84","10,030,649.90","1,202,574.96",116560,11.99,"1,607.73",3.18
2,Home Office,3718,11777,"6,520,538.02","5,859,474.11","690,840.34",68285,11.79,"1,575.97",3.17


In [32]:
customer_performance = (
    df.groupby("Customer Id")
    .agg(
        Segment=("Customer Segment", "first"),
        Orders=("Order Id", "nunique"),
        Net_Revenue=("Order Item Total", "sum"),
        Profit=("Order Profit Per Order", "sum")
    )
    .reset_index()
)

customer_performance = customer_performance.sort_values(
    "Net_Revenue",
    ascending=False
)

display(customer_performance.head(10).round(2))

,Customer Id,Segment,Orders,Net_Revenue,Profit
782,791,Corporate,13,"9,436.61",-866.38
8694,8766,Corporate,11,"8,400.98","1,495.16"
9295,9371,Consumer,11,"8,222.67","1,346.58"
2612,2641,Consumer,11,"8,221.64","2,441.97"
4209,4249,Consumer,13,"8,198.15",439.71
1636,1657,Consumer,11,"8,165.23","2,196.92"
3672,3710,Consumer,14,"8,112.13","1,055.13"
5603,5654,Home Office,15,"7,918.75","1,045.36"
5664,5715,Corporate,12,"7,881.10",-177.45
1271,1288,Home Office,12,"7,869.02",403.11


In [33]:
customer_performance["Revenue Share (%)"] = (
    customer_performance["Net_Revenue"] /
    customer_performance["Net_Revenue"].sum() * 100
)

customer_performance["Cumulative Revenue Share (%)"] = (
    customer_performance["Revenue Share (%)"].cumsum()
)

total_customers = customer_performance["Customer Id"].nunique()

for percentage in [1, 5, 10, 20]:
    customer_count = int(np.ceil(total_customers * percentage / 100))

    revenue_share = (
        customer_performance.head(customer_count)["Net_Revenue"].sum()
        / customer_performance["Net_Revenue"].sum() * 100
    )

    print(
        f"Top {percentage}% of customers "
        f"({customer_count} customers): "
        f"{revenue_share:.2f}% of net revenue"
    )

Top 1% of customers (207 customers): 4.07% of net revenue
Top 5% of customers (1033 customers): 16.50% of net revenue
Top 10% of customers (2066 customers): 29.22% of net revenue
Top 20% of customers (4131 customers): 49.94% of net revenue


In [34]:
customer_performance["Profit-to-Revenue (%)"] = (
    customer_performance["Profit"] /
    customer_performance["Net_Revenue"] * 100
)

customer_performance["Profit-to-Revenue (%)"] = (
    customer_performance["Profit-to-Revenue (%)"].replace(
        [np.inf, -np.inf], np.nan
    )
)

# Customers with positive revenue and negative recorded profit
loss_making_customers = (
    customer_performance[
        (customer_performance["Net_Revenue"] > 0) &
        (customer_performance["Profit"] < 0)
    ]
    .sort_values("Profit")
)

display(loss_making_customers.head(10).round(2))

,Customer Id,Segment,Orders,Net_Revenue,Profit,Revenue Share (%),Cumulative Revenue Share (%),Profit-to-Revenue (%)
1410,1428,Consumer,4,"3,631.04","-3,868.56",0.01,32.21,-106.54
13980,14086,Home Office,1,"1,350.00","-3,442.50",0.00,87.89,-255.00
18003,18109,Corporate,1,"1,320.00","-3,366.00",0.00,88.25,-255.00
14207,14313,Consumer,1,"1,200.00","-3,000.00",0.00,90.35,-250.00
17955,18061,Consumer,1,"1,440.00","-2,592.00",0.00,86.15,-180.00
14007,14113,Consumer,1,"1,500.00","-2,550.00",0.00,84.75,-170.00
13984,14090,Corporate,1,"1,425.00","-2,351.25",0.00,86.50,-165.00
14292,14398,Home Office,1,"1,455.00","-2,328.00",0.00,85.77,-160.00
14200,14306,Home Office,1,"1,425.00","-2,280.00",0.00,86.48,-160.00
14130,14236,Consumer,1,"1,455.00","-2,255.25",0.00,85.76,-155.00


In [35]:
print(
    "Customers with negative recorded profit:",
    len(loss_making_customers)
)

print(
    "Percentage of customers:",
    round(
        len(loss_making_customers) /
        customer_performance["Customer Id"].nunique() * 100,
        2
    ),
    "%"
)

print(
    "Combined recorded loss:",
    round(loss_making_customers["Profit"].sum(), 2)
)

Customers with negative recorded profit: 4069
Percentage of customers: 19.7 %
Combined recorded loss: -1080299.72


In [36]:
loss_segment_analysis = (
    customer_performance
    .assign(
        Loss_Making=lambda x: x["Profit"] < 0
    )
    .groupby("Segment")
    .agg(
        Customers=("Customer Id", "nunique"),
        Loss_Making_Customers=("Loss_Making", "sum"),
        Total_Revenue=("Net_Revenue", "sum"),
        Total_Profit=("Profit", "sum")
    )
    .reset_index()
)

loss_segment_analysis["Loss-Making Customer Rate (%)"] = (
    loss_segment_analysis["Loss_Making_Customers"] /
    loss_segment_analysis["Customers"] * 100
)

display(loss_segment_analysis.round(2))

,Segment,Customers,Loss_Making_Customers,Total_Revenue,Total_Profit,Loss-Making Customer Rate (%)
0,Consumer,10695,2101,"17,164,278.37","2,073,487.67",19.64
1,Corporate,6239,1217,"10,030,649.90","1,202,574.96",19.51
2,Home Office,3718,751,"5,859,474.11","690,840.34",20.20


In [37]:
customer_discount_analysis = (
    df.groupby("Customer Id")
    .agg(
        Net_Revenue=("Order Item Total", "sum"),
        Profit=("Order Profit Per Order", "sum"),
        Total_Discount=("Order Item Discount", "sum"),
        Gross_Sales=("Sales", "sum")
    )
    .reset_index()
)

customer_discount_analysis["Customer Type"] = np.where(
    customer_discount_analysis["Profit"] < 0,
    "Loss-making",
    "Non-loss-making"
)

customer_discount_analysis["Discount Rate (%)"] = (
    customer_discount_analysis["Total_Discount"] /
    customer_discount_analysis["Gross_Sales"] * 100
)

discount_comparison = (
    customer_discount_analysis
    .groupby("Customer Type")
    .agg(
        Customers=("Customer Id", "nunique"),
        Average_Discount_Rate=("Discount Rate (%)", "mean"),
        Median_Discount_Rate=("Discount Rate (%)", "median"),
        Average_Net_Revenue=("Net_Revenue", "mean"),
        Average_Profit=("Profit", "mean")
    )
    .reset_index()
)

display(discount_comparison.round(2))

,Customer Type,Customers,Average_Discount_Rate,Median_Discount_Rate,Average_Net_Revenue,Average_Profit
0,Loss-making,4069,10.16,10.00,"1,571.92",-265.50
1,Non-loss-making,16583,10.14,10.00,"1,607.57",304.36


In [38]:
loss_product_analysis = (
    df.groupby("Category Name")
    .agg(
        Net_Revenue=("Order Item Total", "sum"),
        Profit=("Order Profit Per Order", "sum"),
        Orders=("Order Id", "nunique"),
        Units_Sold=("Order Item Quantity", "sum"),
        Total_Discount=("Order Item Discount", "sum"),
        Gross_Sales=("Sales", "sum")
    )
    .reset_index()
)

loss_product_analysis["Profit-to-Revenue (%)"] = (
    loss_product_analysis["Profit"] /
    loss_product_analysis["Net_Revenue"] * 100
)

loss_product_analysis["Discount Rate (%)"] = (
    loss_product_analysis["Total_Discount"] /
    loss_product_analysis["Gross_Sales"] * 100
)

loss_product_analysis = loss_product_analysis.sort_values(
    "Profit"
)

display(loss_product_analysis.round(2))

,Category Name,Net_Revenue,Profit,Orders,Units_Sold,Total_Discount,Gross_Sales,Profit-to-Revenue (%),Discount Rate (%)
41,Strength Training,"48,813.84",332.31,111,111,"6,081.72","54,895.53",0.68,11.08
7,CDs,"2,750.03",383.85,271,271,309.56,"3,059.59",13.96,10.12
1,As Seen on TV!,"18,519.61",714.43,68,206,"2,078.37","20,597.94",3.86,10.09
5,Books,"11,303.42",883.01,405,405,"1,283.98","12,587.40",7.81,10.20
43,Toys,"5,485.68",900.71,529,529,619.27,"6,104.66",16.42,10.14
2,Baby,"10,957.40","1,525.03",207,207,"1,272.16","12,229.56",13.92,10.40
23,Golf Bags & Carts,"9,403.79","1,810.07",61,61,965.60,"10,369.39",19.25,9.31
4,Basketball,"24,705.33","1,845.67",64,67,"2,394.00","27,099.33",7.47,8.83
33,Men's Clothing,"39,365.71","2,006.04",208,208,"4,491.21","43,856.80",5.10,10.24
45,Video Games,"29,919.20","2,717.52",838,838,"3,392.69","33,310.50",9.08,10.19


In [39]:
weak_categories = [
    "Strength Training",
    "As Seen on  TV!",
    "Men's Clothing",
    "Basketball",
    "Books"
]

weak_category_products = (
    df[df["Category Name"].isin(weak_categories)]
    .groupby(
        ["Category Name", "Product Card Id", "Product Name"]
    )
    .agg(
        Net_Revenue=("Order Item Total", "sum"),
        Profit=("Order Profit Per Order", "sum"),
        Orders=("Order Id", "nunique"),
        Units_Sold=("Order Item Quantity", "sum"),
        Total_Discount=("Order Item Discount", "sum"),
        Gross_Sales=("Sales", "sum")
    )
    .reset_index()
)

weak_category_products["Profit-to-Revenue (%)"] = (
    weak_category_products["Profit"] /
    weak_category_products["Net_Revenue"] * 100
)

weak_category_products["Discount Rate (%)"] = (
    weak_category_products["Total_Discount"] /
    weak_category_products["Gross_Sales"] * 100
)

weak_category_products = weak_category_products.sort_values(
    ["Category Name", "Profit-to-Revenue (%)"]
)

display(weak_category_products.round(2))

,Category Name,Product Card Id,Product Name,Net_Revenue,Profit,Orders,Units_Sold,Total_Discount,Gross_Sales,Profit-to-Revenue (%),Discount Rate (%)
0,As Seen on TV!,359,Nike Men's Free TR 5.0 TB Training Shoe,"18,519.61",714.43,68,206,"2,078.37","20,597.94",3.86,10.09
2,Basketball,60,SOLE E25 Elliptical,"9,414.90",-169.56,10,10,585.00,"9,999.90",-1.80,5.85
3,Basketball,61,Diamondback Girls' Clarity 24 Hybrid Bike 201,"7,624.22",284.42,27,28,775.50,"8,399.72",3.73,9.23
1,Basketball,58,Diamondback Boys' Insight 24 Performance Hybr,"7,666.21","1,730.81",28,29,"1,033.50","8,699.71",22.58,11.88
4,Men's Clothing,1357,Men's gala suit,"39,365.71","2,006.04",208,208,"4,491.21","43,856.80",5.10,10.24
6,Strength Training,208,SOLE E35 Elliptical,"26,409.85",-965.12,15,15,"3,590.00","29,999.85",-3.65,11.97
5,Strength Training,203,GoPro HERO3+ Black Edition Camera,"11,601.68",245.63,32,32,"1,198.00","12,799.68",2.12,9.36
7,Strength Training,216,Yakima DoubleDown Ace Hitch Mount 4-Bike Rack,"10,802.31","1,051.80",64,64,"1,293.72","12,096.00",9.74,10.70


In [40]:
loss_product_ids = [60, 208]

loss_product_transactions = (
    df[df["Product Card Id"].isin(loss_product_ids)]
    .groupby(
        ["Product Card Id", "Product Name"]
    )
    .agg(
        Transactions=("Order Item Id", "count"),
        Orders=("Order Id", "nunique"),
        Units_Sold=("Order Item Quantity", "sum"),
        Gross_Sales=("Sales", "sum"),
        Total_Discount=("Order Item Discount", "sum"),
        Net_Revenue=("Order Item Total", "sum"),
        Recorded_Profit=("Order Profit Per Order", "sum"),
        Average_Discount=("Order Item Discount Rate", "mean"),
        Average_Profit=("Order Profit Per Order", "mean")
    )
    .reset_index()
)

display(loss_product_transactions.round(2))

,Product Card Id,Product Name,Transactions,Orders,Units_Sold,Gross_Sales,Total_Discount,Net_Revenue,Recorded_Profit,Average_Discount,Average_Profit
0,60,SOLE E25 Elliptical,10,10,10,"9,999.90",585.00,"9,414.90",-169.56,0.06,-16.96
1,208,SOLE E35 Elliptical,15,15,15,"29,999.85","3,590.00","26,409.85",-965.12,0.12,-64.34


In [41]:
loss_product_detail = (
    df[df["Product Card Id"].isin(loss_product_ids)]
    [[
        "Order Id",
        "Product Card Id",
        "Product Name",
        "Order Item Quantity",
        "Sales",
        "Order Item Discount",
        "Order Item Discount Rate",
        "Order Item Total",
        "Order Profit Per Order",
        "Order Item Profit Ratio"
    ]]
    .sort_values("Order Profit Per Order")
)

display(loss_product_detail.round(2))

,Order Id,Product Card Id,Product Name,Order Item Quantity,Sales,Order Item Discount,Order Item Discount Rate,Order Item Total,Order Profit Per Order,Order Item Profit Ratio
63990,68859,208,SOLE E35 Elliptical,1,"1,999.99",100.00,0.05,"1,899.99","-4,274.98",-2.25
120716,68775,60,SOLE E25 Elliptical,1,999.99,100.00,0.10,899.99,"-1,844.98",-2.05
148406,68778,208,SOLE E35 Elliptical,1,"1,999.99",300.00,0.15,"1,699.99","-1,246.09",-0.73
159182,68766,208,SOLE E35 Elliptical,1,"1,999.99",320.00,0.16,"1,679.99",-952.56,-0.57
109710,68723,60,SOLE E25 Elliptical,1,999.99,120.00,0.12,879.99,-674.95,-0.77
120715,68804,60,SOLE E25 Elliptical,1,999.99,55.00,0.06,944.99,-177.66,-0.19
148405,68806,208,SOLE E35 Elliptical,1,"1,999.99",260.00,0.13,"1,739.99",34.80,0.02
12606,68703,208,SOLE E35 Elliptical,1,"1,999.99",500.00,0.25,"1,499.99",45.00,0.03
58483,68851,60,SOLE E25 Elliptical,1,999.99,30.00,0.03,969.99,72.75,0.08
173436,68848,208,SOLE E35 Elliptical,1,"1,999.99",140.00,0.07,"1,859.99",139.50,0.08


In [42]:
loss_order_ids = [68859, 68775, 68778, 68766, 68723]

loss_order_investigation = (
    df[df["Order Id"].isin(loss_order_ids)]
    [[
        "Order Id",
        "Product Card Id",
        "Product Name",
        "Order Item Total",
        "Order Profit Per Order",
        "Order Item Profit Ratio",
        "Shipping Mode",
        "Delivery Status",
        "Days for shipping (real)",
        "Days for shipment (scheduled)",
        "Late_delivery_risk",
        "Order Country",
        "Order Region"
    ]]
)

display(loss_order_investigation.round(2))

,Order Id,Product Card Id,Product Name,Order Item Total,Order Profit Per Order,Order Item Profit Ratio,Shipping Mode,Delivery Status,Days for shipping (real),Days for shipment (scheduled),Late_delivery_risk,Order Country,Order Region
63990,68859,208,SOLE E35 Elliptical,"1,899.99","-4,274.98",-2.25,Second Class,Late delivery,5,2,1,Reino Unido,Northern Europe
65566,68859,502,Nike Men's Dri-FIT Victory Golf Polo,75.00,8.48,0.11,Second Class,Late delivery,5,2,1,Reino Unido,Northern Europe
77057,68859,1014,O'Brien Men's Neoprene Life Vest,237.41,-59.35,-0.25,Second Class,Late delivery,5,2,1,Reino Unido,Northern Europe
77077,68778,1014,O'Brien Men's Neoprene Life Vest,141.69,38.26,0.27,Standard Class,Shipping on time,4,4,0,Italia,Southern Europe
77083,68766,1014,O'Brien Men's Neoprene Life Vest,48.98,10.29,0.21,Standard Class,Advance shipping,2,4,0,Francia,Western Europe
109710,68723,60,SOLE E25 Elliptical,879.99,-674.95,-0.77,First Class,Late delivery,2,1,1,Alemania,Western Europe
116752,68723,365,Perfect Fitness Perfect Rip Deck,223.16,25.22,0.11,First Class,Late delivery,2,1,1,Alemania,Western Europe
120716,68775,60,SOLE E25 Elliptical,899.99,"-1,844.98",-2.05,Standard Class,Late delivery,6,4,1,Alemania,Western Europe
120721,68775,191,Nike Men's Free 5.0+ Running Shoe,98.99,27.72,0.28,Standard Class,Late delivery,6,4,1,Alemania,Western Europe
120878,68775,403,Nike Men's CJ Elite 2 TD Football Cleat,107.89,52.87,0.49,Standard Class,Late delivery,6,4,1,Alemania,Western Europe


In [43]:
# Examine the distribution of transaction-level profit ratios

profit_ratio_summary = df["Order Item Profit Ratio"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

display(profit_ratio_summary)

count   180,519.00
mean          0.12
std           0.47
min          -2.75
1%           -1.75
5%           -0.77
25%           0.08
50%           0.27
75%           0.36
95%           0.48
99%           0.50
max           0.50
Name: Order Item Profit Ratio, dtype: float64

In [44]:
# Identify transactions with unusually negative profit ratios

extreme_loss_transactions = (
    df[df["Order Item Profit Ratio"] < -0.5]
    [[
        "Order Id",
        "Product Card Id",
        "Product Name",
        "Order Item Total",
        "Order Profit Per Order",
        "Order Item Profit Ratio",
        "Order Item Discount Rate"
    ]]
    .sort_values("Order Item Profit Ratio")
)

print("Transactions with profit ratio below -0.5:",
      len(extreme_loss_transactions))

display(extreme_loss_transactions.head(20))

Transactions with profit ratio below -0.5: 15695


,Order Id,Product Card Id,Product Name,Order Item Total,Order Profit Per Order,Order Item Profit Ratio,Order Item Discount Rate
85751,2774,1004,Field & Stream Sportsman 16 Gun Fire Safe,391.98,"-1,077.95",-2.75,0.02
113763,4125,1014,O'Brien Men's Neoprene Life Vest,43.98,-120.95,-2.75,0.12
938,11968,365,Perfect Fitness Perfect Rip Deck,299.95,-824.86,-2.75,0.00
169549,31506,567,adidas Men's Germany Black Crest Away Tee,22.50,-61.88,-2.75,0.10
169912,23137,502,Nike Men's Dri-FIT Victory Golf Polo,212.50,-584.38,-2.75,0.15
50771,59691,730,LIJA Women's Argyle Golf Polo,64.00,-176.00,-2.75,0.20
102099,23242,1073,Pelican Sunstream 100 Kayak,181.99,-500.47,-2.75,0.09
49119,32124,403,Nike Men's CJ Elite 2 TD Football Cleat,106.59,-293.13,-2.75,0.18
21075,28141,403,Nike Men's CJ Elite 2 TD Football Cleat,116.99,-321.73,-2.75,0.10
124552,16469,502,Nike Men's Dri-FIT Victory Golf Polo,83.00,-228.25,-2.75,0.17


In [45]:
profit_relationship = df[
    [
        "Order Profit Per Order",
        "Order Item Profit Ratio",
        "Order Item Total"
    ]
].corr()

display(profit_relationship.round(3))

,Order Profit Per Order,Order Item Profit Ratio,Order Item Total
Order Profit Per Order,1.00,0.82,0.13
Order Item Profit Ratio,0.82,1.00,-0.00
Order Item Total,0.13,-0.00,1.00


In [46]:
# Classify transactions based on recorded profit ratio

df["Profit_Ratio_Group"] = pd.cut(
    df["Order Item Profit Ratio"],
    bins=[
        -float("inf"),
        -1,
        -0.5,
        0,
        0.1,
        0.25,
        0.5
    ],
    labels=[
        "Below -100%",
        "-100% to -50%",
        "-50% to 0%",
        "0% to 10%",
        "10% to 25%",
        "25% to 50%"
    ],
    include_lowest=True
)

profit_ratio_distribution = (
    df.groupby("Profit_Ratio_Group", observed=False)
    .agg(
        Transactions=("Order Item Id", "count"),
        Net_Revenue=("Order Item Total", "sum"),
        Recorded_Profit=("Order Profit Per Order", "sum")
    )
    .reset_index()
)

profit_ratio_distribution["Transaction_Share_%"] = (
    profit_ratio_distribution["Transactions"]
    / len(df) * 100
)

display(profit_ratio_distribution.round(2))

,Profit_Ratio_Group,Transactions,Net_Revenue,Recorded_Profit,Transaction_Share_%
0,Below -100%,6301,"1,169,052.21","-2,062,101.22",3.49
1,-100% to -50%,9502,"1,734,770.30","-1,257,931.71",5.26
2,-50% to 0%,19158,"3,475,755.01","-563,514.42",10.61
3,0% to 10%,17976,"3,277,066.01","194,450.47",9.96
4,10% to 25%,28288,"5,194,301.38","840,745.42",15.67
5,25% to 50%,99294,"18,203,457.47","6,815,254.43",55.00


In [47]:
# Investigate repeated extreme profit ratios

extreme_ratio_check = (
    df[df["Order Item Profit Ratio"] <= -2]
    .groupby("Order Item Profit Ratio")
    .agg(
        Transactions=("Order Item Id", "count"),
        Products=("Product Card Id", "nunique"),
        Orders=("Order Id", "nunique"),
        Net_Revenue=("Order Item Total", "sum"),
        Recorded_Profit=("Order Profit Per Order", "sum")
    )
    .reset_index()
    .sort_values("Order Item Profit Ratio")
)

display(extreme_ratio_check.round(2))

,Order Item Profit Ratio,Transactions,Products,Orders,Net_Revenue,Recorded_Profit
0,-2.75,72,18,72,"13,397.72","-36,843.88"
1,-2.70,252,41,252,"46,106.87","-124,489.19"
2,-2.65,90,21,90,"16,663.69","-44,159.02"
3,-2.60,181,34,181,"31,771.69","-82,606.74"
4,-2.55,234,32,223,"46,221.23","-117,864.75"
5,-2.50,180,30,180,"32,062.44","-80,156.61"
6,-2.45,18,10,18,"3,122.86","-7,651.03"
7,-2.35,36,15,36,"6,339.64","-14,898.23"
8,-2.30,18,8,18,"1,958.25","-4,503.99"
9,-2.25,36,12,36,"8,252.70","-18,568.66"


In [48]:
extreme_ratio_check = (
    df[df["Order Item Profit Ratio"] < -1]
    .groupby("Order Item Profit Ratio")
    .agg(
        Transactions=("Order Item Id", "count"),
        Products=("Product Card Id", "nunique"),
        Orders=("Order Id", "nunique"),
        Net_Revenue=("Order Item Total", "sum"),
        Recorded_Profit=("Order Profit Per Order", "sum")
    )
    .reset_index()
    .sort_values("Order Item Profit Ratio")
)

print("Transactions with profit ratio below -100%:",
      extreme_ratio_check["Transactions"].sum())

print("Unique products affected:",
      df.loc[
          df["Order Item Profit Ratio"] < -1,
          "Product Card Id"
      ].nunique())

print("Unique orders affected:",
      df.loc[
          df["Order Item Profit Ratio"] < -1,
          "Order Id"
      ].nunique())

display(extreme_ratio_check.round(2))

Transactions with profit ratio below -100%: 6301
Unique products affected: 114
Unique orders affected: 5548


,Order Item Profit Ratio,Transactions,Products,Orders,Net_Revenue,Recorded_Profit
0,-2.75,72,18,72,"13,397.72","-36,843.88"
1,-2.70,252,41,252,"46,106.87","-124,489.19"
2,-2.65,90,21,90,"16,663.69","-44,159.02"
3,-2.60,181,34,181,"31,771.69","-82,606.74"
4,-2.55,234,32,223,"46,221.23","-117,864.75"
5,-2.50,180,30,180,"32,062.44","-80,156.61"
6,-2.45,18,10,18,"3,122.86","-7,651.03"
7,-2.35,36,15,36,"6,339.64","-14,898.23"
8,-2.30,18,8,18,"1,958.25","-4,503.99"
9,-2.25,36,12,36,"8,252.70","-18,568.66"


In [49]:
# Check whether the recorded profit ratio is related
# to recorded profit and net item revenue through a consistent formula.

df["Calculated_Profit_Ratio"] = (
    df["Order Profit Per Order"] / df["Order Item Total"]
)

df["Profit_Ratio_Difference"] = (
    df["Order Item Profit Ratio"] -
    df["Calculated_Profit_Ratio"]
)

print(
    "Correlation between recorded and calculated ratios:",
    df[
        ["Order Item Profit Ratio", "Calculated_Profit_Ratio"]
    ].corr().iloc[0, 1].round(4)
)

print(
    "Mean absolute difference:",
    df["Profit_Ratio_Difference"].abs().mean().round(4)
)

print(
    "Median absolute difference:",
    df["Profit_Ratio_Difference"].abs().median().round(4)
)

display(
    df[
        [
            "Order Item Total",
            "Order Profit Per Order",
            "Order Item Profit Ratio",
            "Calculated_Profit_Ratio",
            "Profit_Ratio_Difference"
        ]
    ]
    .sort_values("Profit_Ratio_Difference",
                 key=lambda x: x.abs(),
                 ascending=False)
    .head(10)
    .round(3)
)

Correlation between recorded and calculated ratios: 1.0
Mean absolute difference: 0.0012
Median absolute difference: 0.0


,Order Item Total,Order Profit Per Order,Order Item Profit Ratio,Calculated_Profit_Ratio,Profit_Ratio_Difference
52582,10.39,-4.93,-0.48,-0.47,-0.01
23153,10.84,4.06,0.38,0.38,0.01
52577,11.08,1.38,0.13,0.12,0.01
52433,11.08,1.38,0.13,0.12,0.01
32709,8.49,3.18,0.38,0.38,0.01
52263,9.58,3.11,0.33,0.33,0.01
162559,11.18,3.63,0.33,0.33,0.01
52368,10.04,0.75,0.08,0.07,0.01
52548,10.04,0.75,0.08,0.07,0.01
52427,9.23,-3.92,-0.43,-0.42,-0.01


In [50]:
regional_profitability = (
    df.groupby("Order Region")
    .agg(
        Orders=("Order Id", "nunique"),
        Transactions=("Order Item Id", "count"),
        Net_Revenue=("Order Item Total", "sum"),
        Recorded_Profit=("Order Profit Per Order", "sum"),
        Negative_Transactions=(
            "Order Profit Per Order",
            lambda x: (x < 0).sum()
        )
    )
    .reset_index()
)

regional_profitability["Profit_to_Revenue_%"] = (
    regional_profitability["Recorded_Profit"]
    / regional_profitability["Net_Revenue"] * 100
)

regional_profitability["Negative_Transaction_Rate_%"] = (
    regional_profitability["Negative_Transactions"]
    / regional_profitability["Transactions"] * 100
)

regional_profitability = regional_profitability.sort_values(
    "Negative_Transaction_Rate_%",
    ascending=False
)

display(regional_profitability.round(2))

,Order Region,Orders,Transactions,Net_Revenue,Recorded_Profit,Negative_Transactions,Profit_to_Revenue_%,Negative_Transaction_Rate_%
2,Central Africa,556,1677,"292,912.56","33,447.27",334,11.42,19.92
8,Eastern Europe,1292,3920,"696,307.19","79,717.05",773,11.45,19.72
7,Eastern Asia,3318,7280,"1,334,313.41","147,368.01",1397,11.04,19.19
11,Oceania,4362,10148,"1,809,996.55","201,478.02",1944,11.13,19.16
9,North Africa,1064,3232,"572,241.97","64,599.86",617,11.29,19.09
13,South Asia,3335,7731,"1,397,364.57","165,703.90",1469,11.86,19.00
19,West Africa,1223,3696,"654,168.03","80,030.27",699,12.23,18.91
21,West of USA,2667,7993,"1,412,253.60","164,940.66",1511,11.68,18.90
18,US Center,1935,5887,"1,034,129.43","131,094.23",1111,12.68,18.87
22,Western Europe,10010,27109,"5,296,002.89","625,446.08",5104,11.81,18.83


In [51]:
df["Order Region"] = (
    df["Order Region"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)